# Hybrid Approach — Dependency Parsing + Embedding + HDBSCAN Clustering

## What this notebook does
Implements the **Hybrid Approach**: injecting event-level structure into the pipeline
via dependency parsing before embedding. Rather than embedding full article text
(which failed in `01_direct_approach.ipynb`), we first extract Subject-Verb-Object
triples from each article using spaCy, then embed those structured representations
using three models: **E5**, **SecureBERT**, and **GloVe**.

## Why this works where the Direct Approach failed
Full-text embeddings capture broad thematic similarity — all cybersecurity articles
look like each other at the corpus level. Dependency parsing forces the model to
compare relational structure (*who did what to whom*) rather than surface vocabulary.
This acts as pseudo-annotation: event-level structure without the cost of manual labeling.

## What this notebook concludes
The Hybrid E5 pipeline produces **152–269 coherent event-level clusters**, including
clusters that correctly identify real-world incidents across multiple news outlets.
This validates the feasibility of fully unsupervised cybersecurity event extraction
when structural signals are incorporated prior to embedding.

## Pipeline
```
Raw Articles
    → spaCy Dependency Parsing (SVO triple extraction)
    → Event Template Construction (semicolon-joined triples per article)
    → Embedding:
        ├── E5  (intfloat/e5-base-v2)         → 768-dim, article-level
        ├── SecureBERT (ehsanaghaei/SecureBERT)→ 768-dim, triple-averaged
        └── GloVe (glove.6B.300d)              → 300-dim, triple-averaged
    → L2 Normalization
    → HDBSCAN Clustering
    → UMAP Visualization
```

## Models
| Model | Type | Input | Pooling | Embedding Dim |
|-------|------|-------|---------|---------------|
| `intfloat/e5-base-v2` | General-purpose, instruction-tuned | Full event template | Average pool + L2 norm | 768 |
| `ehsanaghaei/SecureBERT` | Cybersecurity domain-adapted | Per-triple | 4-layer masked mean → article mean | 768 |
| GloVe `glove.6B.300d` | Static word vectors | Per-triple word tokens | Word mean → triple mean → article mean | 300 |

## 1. Imports

In [ ]:
import re
import string
import unicodedata
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import Tensor
import hdbscan
import umap
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from transformers import AutoTokenizer, AutoModel, RobertaTokenizerFast, RobertaModel
from sklearn.preprocessing import normalize
from sklearn.metrics import silhouette_score
from joblib import Parallel, delayed
from sklearn.model_selection import ParameterGrid
from gensim.models import KeyedVectors
from gensim.scripts.glove2word2vec import glove2word2vec
from gensim.test.utils import datapath, get_tmpfile
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Dependency Parsing — SVO Triple Extraction

This section transforms raw cybersecurity articles into structured event representations
using spaCy's transformer-based dependency parser (`en_core_web_trf`).

Each sentence is converted into a Subject-Verb-Object triple. For example:
> *"LockBit ransomware attacked a hospital in Germany"*
> → `(subject=lockbit, verb=attack, object=hospital)`

Partial triples where subject or object cannot be identified are retained with
a `<none>` placeholder to maximize recall — sparse events reported by only 2–3
outlets would otherwise be lost entirely.

All tokens are normalized: lemmatized, lowercased, stop words and punctuation removed.
CVE identifiers (e.g., `CVE-2025-1234`) are preserved via regex before normalization.

All triples for each article are concatenated with semicolons into a single
**event template string**, which serves as the input to all three embedding models.

**Output:** `article_event_templates.csv` with columns `article_id` and `event_text`.

In [ ]:
import os, re, spacy
import pandas as pd
import numpy as np
from collections import defaultdict

# ── Constants ─────────────────────────────────────────────────────────────
# CVE identifiers follow a strict format — preserve them exactly
CVE_RE = re.compile(r"\bCVE-\d{4}-\d+\b", flags=re.IGNORECASE)

# Pronouns that never carry meaningful event information
PRONOUNS = {
    "it", "they", "them", "their", "its",
    "this", "that", "those", "these",
    "who", "which", "we", "you", "i"
}


def load_csv_robust(path):
    """Load CSV with UTF-8, cp1252, and latin1 encoding fallbacks."""
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            pass
    raise RuntimeError(f"Could not read {path} with utf-8 / cp1252 / latin1")


def clean_entity_text(tok):
    """
    Normalize a token to its canonical event representation.
    CVE identifiers are uppercased and preserved exactly.
    All other tokens are lemmatized, lowercased, and stripped of noise characters.
    """
    if isinstance(tok, str):
        return tok
    txt = tok.text
    if CVE_RE.search(txt):
        return txt.upper()
    lemma = tok.lemma_.lower().strip()
    lemma = re.sub(r"[^a-z0-9\-\.]+", " ", lemma)
    return re.sub(r"\s+", " ", lemma).strip()


def is_good_entity(tok):
    """
    Filter out tokens that carry no meaningful event information.
    Rejects stop words (unless proper nouns), pronouns, and single characters.
    """
    if tok.is_stop and tok.pos_ != "PROPN":
        return False
    if tok.lemma_.lower() in PRONOUNS:
        return False
    if len(tok.lemma_) <= 1 and not CVE_RE.search(tok.text):
        return False
    return True


def extract_svo_from_doc(doc):
    """
    Extract SVO triples from a parsed spaCy document.

    For each sentence, finds verbs heading clauses (ROOT, conj, xcomp, ccomp),
    then extracts their syntactic subjects and objects.
    Partial triples missing subject or object are retained with '<none>'
    to maximize recall for sparsely-reported events.
    """
    triples = []
    for sent_id, sent in enumerate(doc.sents):
        verbs = [
            t for t in sent
            if t.pos_ in ("VERB", "AUX")
            and t.dep_ in ("ROOT", "conj", "xcomp", "ccomp")
        ]
        for v in verbs:
            subs = [w for w in v.children if w.dep_ in ("nsubj", "nsubjpass") and is_good_entity(w)]
            objs = [w for w in v.children if w.dep_ in ("dobj", "attr", "dative", "oprd") and is_good_entity(w)]
            for prep in [w for w in v.children if w.dep_ == "prep"]:
                objs.extend([w for w in prep.children if w.dep_ == "pobj" and is_good_entity(w)])

            if not (subs or objs):
                continue

            if not subs: subs = ["<none>"]
            if not objs: objs = ["<none>"]

            voice = "passive" if any(
                s != "<none>" and hasattr(s, "dep_") and s.dep_ == "nsubjpass"
                for s in subs
            ) else "active"
            v_norm = v.lemma_.lower()

            for s in subs:
                s_norm = clean_entity_text(s) if s != "<none>" else "<none>"
                for o in objs:
                    o_norm = clean_entity_text(o) if o != "<none>" else "<none>"
                    triples.append({
                        "sent_id": sent_id,
                        "subject": s_norm, "verb": v_norm, "object": o_norm,
                        "voice": voice,
                        "subject_span": s.text if s != "<none>" else "<none>",
                        "verb_span": v.text,
                        "object_span": o.text if o != "<none>" else "<none>"
                    })
    return triples


def triples_to_event_template(triples_df, n_total):
    """
    Convert per-sentence SVO triples to article-level event template strings.
    Each triple becomes a 'subject verb object' mini-event.
    All mini-events per article are deduplicated and joined with ' ; '.
    Articles with no triples receive 'EVENT: <none>' placeholder.
    """
    def norm_piece(x):
        if not isinstance(x, str) or x.strip() in ("", "<none>"):
            return ""
        if CVE_RE.fullmatch(x.strip()):
            return x.upper()
        return x.lower().strip()

    df = triples_df.copy()
    df["subject_n"] = df["subject"].apply(norm_piece)
    df["verb_n"]    = df["verb"].apply(norm_piece)
    df["object_n"]  = df["object"].apply(norm_piece)
    df["mini"]      = df.apply(
        lambda r: " ".join(p for p in [r["subject_n"], r["verb_n"], r["object_n"]] if p),
        axis=1
    )

    by_article = defaultdict(list)
    for aid, chunk in df.groupby("article_id"):
        texts = sorted(set(t for t in chunk["mini"].tolist() if t))
        if texts:
            by_article[int(aid)] = texts

    rows = []
    for aid in range(n_total):
        events = by_article.get(aid, [])
        rows.append({"article_id": aid, "event_text": " ; ".join(events) if events else "EVENT: <none>"})

    return pd.DataFrame(rows)


# ── Load pre-computed event templates ─────────────────────────────────────────
# The full parsing pipeline was run on Google Colab (en_core_web_trf)
# and outputs saved to article_event_templates.csv.
#
# To re-run parsing from scratch:
#   python src/dependency_parser.py \
#       --input data/full_clean.csv \
#       --output data/article_event_templates.csv

templates_df = load_csv_robust("../data/article_event_templates.csv")

print(f"Articles with event templates: {len(templates_df):,}")
print(f"Columns: {list(templates_df.columns)}")
print("\nSample event template:")
print(templates_df['event_text'].iloc[0])

### 2.1 Triple Splitting Utility

The event template string is a semicolon-joined sequence of SVO triples.
For SecureBERT and GloVe, we embed each triple independently before averaging
to an article-level representation. This function handles the splitting.

In [ ]:
def split_event_template(df, event_col="event_text"):
    """
    Explode semicolon-joined event templates into one row per triple.

    Args:
        df: DataFrame with article_id and event_text columns
        event_col: Column containing semicolon-joined SVO triples

    Returns:
        DataFrame with one row per triple: article_id, triple_id, split_text
    """
    df = df.copy()
    # Strip outer braces if present, split on semicolons
    df["chunks"] = df[event_col].str.strip("{}").str.split(r"\s*;\s*", regex=True)

    exploded = (
        df[["article_id", "chunks"]]
        .explode("chunks")
        .rename(columns={"chunks": "split_text"})
        .dropna()
        .assign(triple_id=lambda d: d.groupby(level=0).cumcount())
        .reset_index(drop=True)
    )

    return exploded[["article_id", "triple_id", "split_text"]]


triples_df = split_event_template(templates_df)
print(f"Total triples extracted: {len(triples_df):,}")
print(f"Average triples per article: {len(triples_df)/len(templates_df):.1f}")
triples_df.head(5)

## 3. E5 Embedding

E5 (`intfloat/e5-base-v2`) is embedded differently from SecureBERT and GloVe.
Rather than embedding individual triples and averaging, E5 embeds the **full
event template string** for each article in a single pass.

**Why:** E5 is instruction-tuned for retrieval and clustering — its training
objective is optimized for document-level semantic comparison. Feeding it the
full structured template (all triples joined) produces more directionally
consistent embeddings than averaging independently embedded triples.

**Instruction prefix:** E5 requires a `"query: "` prefix. This is a hard
requirement of its instruction-tuned architecture — omitting it degrades
embedding quality.

**Chunking:** Articles whose event templates exceed 512 tokens are split into
chunks, each independently embedded, then averaged into one document vector.

In [ ]:
E5_MODEL = "intfloat/e5-base-v2"

e5_tokenizer = AutoTokenizer.from_pretrained(E5_MODEL)
e5_model = AutoModel.from_pretrained(E5_MODEL)
e5_model.eval()
e5_model.to(device)

print(f"E5 loaded on {device}")

In [ ]:
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    """Masked average pool over token dimension, ignoring padding positions."""
    masked = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return masked.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


def embed_texts_e5(texts, tokenizer, model, batch_size=16, max_length=512):
    """
    Embed a list of event template strings using E5.

    Each text is prefixed with 'query: ' per E5's instruction-tuning requirement.
    Texts exceeding 512 tokens are chunked, each chunk embedded independently,
    then averaged into a single document vector.

    Args:
        texts: List of event template strings (one per article)
        tokenizer: E5 tokenizer
        model: E5 model
        batch_size: Articles per batch
        max_length: Token limit per chunk

    Returns:
        np.ndarray of shape (n_articles, 768)
    """
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding with E5"):
        batch_texts = texts[i:i + batch_size]
        batch_embs = []

        for t in batch_texts:
            prefixed = "query: " + t  # required E5 instruction prefix
            tokenized = tokenizer(prefixed, add_special_tokens=False)["input_ids"]

            # Pre-chunk texts that exceed the token limit
            if len(tokenized) > max_length:
                chunks = [tokenized[j:j + max_length] for j in range(0, len(tokenized), max_length)]
                chunk_texts = [tokenizer.decode(c) for c in chunks]
            else:
                chunk_texts = [prefixed]

            encoded = tokenizer(
                chunk_texts,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            with torch.no_grad():
                outputs = e5_model(**encoded)

            emb = average_pool(outputs.last_hidden_state, encoded["attention_mask"])
            emb = F.normalize(emb, p=2, dim=1)  # L2 normalize each chunk

            # Average across chunks → single document vector
            doc_emb = emb.mean(dim=0, keepdim=True)
            batch_embs.append(doc_emb)

        batch_embs = torch.cat(batch_embs, dim=0)
        embeddings.append(batch_embs)

    return torch.cat(embeddings, dim=0).cpu().numpy()

In [ ]:
# NOTE: Compute-heavy. Uncomment to re-run from scratch:

# texts = templates_df["event_text"].fillna("").astype(str).str.strip().tolist()
# e5_embs = embed_texts_e5(texts, e5_tokenizer, e5_model, batch_size=16)
#
# # Save as both .npy and structured DataFrame
# np.save("../results/e5_hybrid_embeddings.npy", e5_embs)
# emb_cols = [f"v{i}" for i in range(e5_embs.shape[1])]
# e5_df = pd.concat([
#     templates_df[["article_id", "event_text"]].reset_index(drop=True),
#     pd.DataFrame(e5_embs, columns=emb_cols)
# ], axis=1)
# e5_df.to_csv("../results/e5_hybrid_embeddings.csv", index=False)
# print(f"Saved {len(e5_df):,} E5 hybrid embeddings")

# Load pre-computed embeddings (generated on CU Boulder Alpine HPC)
e5_embs = np.load("../results/e5_hybrid_embeddings.npy")
e5_meta = pd.read_csv("../results/e5_hybrid_embeddings.csv", usecols=["article_id", "event_text"])

print(f"E5 hybrid embeddings loaded: {e5_embs.shape}")
print(f"Articles: {len(e5_meta):,} | Embedding dim: {e5_embs.shape[1]}")

## 4. SecureBERT Embedding

SecureBERT embeds each SVO triple independently using 4-layer masked mean pooling,
then averages all triple embeddings to produce a single article-level vector.

**Why per-triple rather than full template:** SecureBERT is optimized for
sentence-level cybersecurity text. Embedding individual triples and averaging
preserves fine-grained relational signal from each event mention, rather than
diluting it across the full concatenated template string.

In [ ]:
SECUREBERT_MODEL = "ehsanaghaei/SecureBERT"

sb_tokenizer = RobertaTokenizerFast.from_pretrained(SECUREBERT_MODEL)
sb_model = RobertaModel.from_pretrained(SECUREBERT_MODEL, add_pooling_layer=False).eval()
sb_model.to(device)

print(f"SecureBERT loaded on {device}")

In [ ]:
def securebert_embed_triple(text):
    """
    Embed a single SVO triple string using SecureBERT.

    Uses 4-layer masked mean pooling:
    - Averages the final 4 transformer hidden layers (stabilizes signal
      for models without explicit sentence embedding objectives)
    - Then masked mean pools over token positions

    Returns:
        np.ndarray of shape (768,)
    """
    batch = sb_tokenizer(text, return_tensors="pt", padding=False, truncation=False)
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        out = sb_model(**batch, output_hidden_states=True)

    # Average last 4 hidden layers → [1, T, H]
    last4 = out.hidden_states[-4:]
    token_reps = torch.stack(last4, dim=0).mean(dim=0)

    # Masked mean pool over token positions
    attn = batch["attention_mask"].unsqueeze(-1).to(token_reps.dtype)
    sent_emb = (token_reps * attn).sum(dim=1) / attn.sum(dim=1).clamp(min=1e-9)

    return sent_emb.squeeze(0).cpu().numpy()


def parse_embedding(x):
    """Safely convert stored embedding to numpy array regardless of format."""
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, str):
        cleaned = x.strip().replace("tensor(", "").rstrip(")")
        return np.array(eval(cleaned), dtype=float)
    raise TypeError(f"Unknown embedding type: {type(x)}")


def aggregate_securebert(triples_df):
    """
    Embed each triple and average to article level.

    Returns:
        DataFrame with article_id and article_securebert_mean columns
    """
    triples_df = triples_df.copy()
    triples_df["triple_embeddings"] = triples_df["split_text"].apply(securebert_embed_triple)
    triples_df["triple_embeddings"] = triples_df["triple_embeddings"].apply(parse_embedding)

    def mean_vec(series):
        arrs = [np.asarray(x, dtype=float) for x in series]
        return np.mean(np.vstack(arrs), axis=0)

    return (
        triples_df.groupby("article_id")["triple_embeddings"]
        .apply(mean_vec)
        .reset_index(name="article_securebert_mean")
    )

In [ ]:
# NOTE: Compute-heavy. Uncomment to re-run from scratch:

# sb_article_df = aggregate_securebert(triples_df)
# sb_article_df.to_csv("../results/securebert_hybrid_embeddings.csv", index=False)
# print(f"Saved {len(sb_article_df):,} SecureBERT hybrid embeddings")

# Load pre-computed embeddings (generated on CU Boulder Alpine HPC)
sb_article_df = pd.read_csv("../results/securebert_hybrid_embeddings.csv")
sb_article_df["article_securebert_mean"] = sb_article_df["article_securebert_mean"].apply(
    lambda x: np.fromstring(x.strip("[]"), sep=" ")
)

print(f"SecureBERT hybrid embeddings loaded: {len(sb_article_df):,} articles")
print(f"Embedding dimension: {len(sb_article_df['article_securebert_mean'].iloc[0])}")

## 5. GloVe Embedding

GloVe provides a complementary baseline using static 300-dimensional word vectors
trained on Wikipedia 2014 and Gigaword. Unlike E5 and SecureBERT, GloVe has no
contextual awareness — each word maps to a fixed vector regardless of surrounding text.

**Pipeline:** For each article, each SVO triple is tokenized into words, each word
is looked up in the GloVe vocabulary, word vectors are averaged into a triple vector,
and all triple vectors are averaged into a single 300-dim article vector.

Words not found in the GloVe vocabulary are excluded. CVE identifiers and
specialized cybersecurity terms have limited GloVe coverage — this is a
known limitation relative to the domain-adapted models.

**Note:** GloVe embeddings are 300-dimensional vs. 768-dimensional for E5
and SecureBERT. This difference is handled consistently in normalization
and visualization.

In [ ]:
# Load GloVe vectors via Gensim
# Download glove.6B.300d.txt from https://nlp.stanford.edu/projects/glove/
GLOVE_PATH = "../data/glove.6B.300d.txt"  # update path if needed
GLOVE_W2V_PATH = get_tmpfile("glove.6B.300d.word2vec.txt")

glove2word2vec(GLOVE_PATH, GLOVE_W2V_PATH)
glove_model = KeyedVectors.load_word2vec_format(GLOVE_W2V_PATH)

print(f"GloVe vocabulary size: {len(glove_model.key_to_index):,}")
print(f"Embedding dimension: {glove_model.vector_size}")

In [ ]:
def strip_unicode_punct(s):
    """Remove all Unicode punctuation characters from a string."""
    return "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))


def embed_article_glove(event_text, glove_model):
    """
    Embed a full event template string using GloVe.

    Pipeline per article:
      event_text → split by ';' → tokenize each triple → GloVe lookup
      → average word vectors → average triple vectors → article vector

    Words not in GloVe vocabulary are excluded.
    Triples with no valid word vectors are excluded.

    Returns:
        np.ndarray of shape (300,) or None if no valid tokens found
    """
    triples = [g.strip() for g in str(event_text).split(";") if g.strip()]
    triple_means = []

    for triple in triples:
        raw_tokens = triple.split()
        # Normalize tokens: remove punctuation, lowercase
        cleaned_tokens = [
            strip_unicode_punct(tok).casefold()
            for tok in raw_tokens
        ]
        # Look up GloVe vectors, skip OOV tokens
        valid_vecs = [
            glove_model[tok]
            for tok in cleaned_tokens
            if tok and tok in glove_model.key_to_index
        ]
        if not valid_vecs:
            continue  # skip triples with no GloVe coverage
        triple_means.append(np.stack(valid_vecs).mean(axis=0))

    if not triple_means:
        return None

    return np.stack(triple_means).mean(axis=0)  # average across triples


def aggregate_glove(templates_df, glove_model):
    """
    Apply GloVe embedding to all articles and return article-level DataFrame.

    Returns:
        DataFrame with article_id and article_glove_mean columns
    """
    records = []
    for _, row in tqdm(templates_df.iterrows(), total=len(templates_df), desc="GloVe embedding"):
        vec = embed_article_glove(row["event_text"], glove_model)
        if vec is not None:
            records.append({"article_id": row["article_id"], "article_glove_mean": vec})

    return pd.DataFrame(records)

In [ ]:
# NOTE: Compute-heavy. Uncomment to re-run from scratch:

# glove_article_df = aggregate_glove(templates_df, glove_model)
# glove_article_df.to_csv("../results/glove_hybrid_embeddings.csv", index=False)
# print(f"Saved {len(glove_article_df):,} GloVe hybrid embeddings")

# Load pre-computed embeddings (generated on CU Boulder Alpine HPC)
glove_article_df = pd.read_csv("../results/glove_hybrid_embeddings.csv")
glove_article_df["article_glove_mean"] = glove_article_df["article_glove_mean"].apply(
    lambda x: np.fromstring(x.strip("[]"), sep=" ")
)

print(f"GloVe hybrid embeddings loaded: {len(glove_article_df):,} articles")
print(f"Embedding dimension: {len(glove_article_df['article_glove_mean'].iloc[0])}")

## 6. HDBSCAN Clustering

HDBSCAN is applied identically across all three models. All embeddings are
L2-normalized before clustering regardless of dimensionality (768 for E5/SecureBERT,
300 for GloVe), ensuring Euclidean distance reflects directional similarity.

Final parameters were selected via grid search (see Section 7):
- `min_cluster_size = 4`
- `min_samples = 1`
- `cluster_selection_epsilon = 0.0001`

In [ ]:
# Final parameters selected via grid search
MIN_CLUSTER_SIZE = 4
MIN_SAMPLES = 1
CLUSTER_SELECTION_EPS = 0.0001


def run_hdbscan(embedding_col, article_df, min_cluster_size, min_samples, eps):
    """
    Run HDBSCAN on article-level embeddings.

    Args:
        embedding_col: Column name containing embedding arrays
        article_df: DataFrame with article_id and embedding column
        min_cluster_size, min_samples, eps: HDBSCAN parameters

    Returns:
        result_df: article_df with cluster_id and membership_prob columns
        X_norm: L2-normalized embedding matrix
    """
    X = np.vstack(article_df[embedding_col].values)
    X_norm = normalize(X, norm="l2")

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        cluster_selection_epsilon=eps,
        metric="euclidean",
        gen_min_span_tree=True
    )
    clusterer.fit(X_norm)

    result_df = article_df.copy()
    result_df["cluster_id"] = clusterer.labels_
    result_df["membership_prob"] = clusterer.probabilities_

    return result_df, X_norm


def print_clustering_summary(name, result_df, X_norm):
    """Print clustering diagnostics."""
    labels = result_df["cluster_id"].values
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_frac = (labels == -1).sum() / len(labels)
    sil = silhouette_score(X_norm, labels) if n_clusters > 1 else None

    print(f"\n{name}")
    print(f"  Clusters:          {n_clusters}")
    print(f"  Noise fraction:    {noise_frac:.4f}")
    print(f"  Silhouette score:  {sil:.4f}" if sil else "  Silhouette score:  N/A")

    return n_clusters, noise_frac, sil

In [ ]:
# Stack E5 embeddings from column format
emb_cols = [f"v{i}" for i in range(768)]
e5_article_df = e5_meta.copy()
e5_article_df["embedding"] = list(e5_embs)

# Run clustering — all three models
e5_clustered, e5_X_norm = run_hdbscan(
    "embedding", e5_article_df, MIN_CLUSTER_SIZE, MIN_SAMPLES, CLUSTER_SELECTION_EPS
)
e5_n, e5_noise, e5_sil = print_clustering_summary("Hybrid E5", e5_clustered, e5_X_norm)

sb_clustered, sb_X_norm = run_hdbscan(
    "article_securebert_mean", sb_article_df, MIN_CLUSTER_SIZE, MIN_SAMPLES, CLUSTER_SELECTION_EPS
)
sb_n, sb_noise, sb_sil = print_clustering_summary("Hybrid SecureBERT", sb_clustered, sb_X_norm)

glove_clustered, glove_X_norm = run_hdbscan(
    "article_glove_mean", glove_article_df, MIN_CLUSTER_SIZE, MIN_SAMPLES, CLUSTER_SELECTION_EPS
)
glove_n, glove_noise, glove_sil = print_clustering_summary("Hybrid GloVe", glove_clustered, glove_X_norm)

## 7. Parameter Tuning

Grid search was run on Alpine HPC across all three models using the same
parameter grid applied in the Direct Approach for consistent comparison.

| Parameter | Values Tested |
|-----------|---------------|
| `min_cluster_size` | 2, 4 |
| `min_samples` | 1, 2, 3, 4 |
| `cluster_selection_epsilon` | 0.0001, 0.001, 0.01, 0.1 |

In [ ]:
def evaluate_hdbscan(X, min_cluster_size, min_samples, eps):
    """Evaluate a single HDBSCAN parameter combination."""
    try:
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            cluster_selection_epsilon=eps,
            metric="euclidean",
            gen_min_span_tree=True
        ).fit(X)

        labels = clusterer.labels_
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_frac = np.sum(labels == -1) / len(labels)
        silhouette = silhouette_score(X, labels) if n_clusters > 1 else None

        return {
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "eps": eps,
            "n_clusters": n_clusters,
            "noise_frac": round(noise_frac, 3),
            "silhouette": round(silhouette, 4) if silhouette else None
        }
    except Exception as e:
        return {"min_cluster_size": min_cluster_size, "min_samples": min_samples,
                "eps": eps, "error": str(e)}


# To re-run grid search for any model, uncomment:
# param_grid = {
#     "MIN_CLUSTER_SIZE": [2, 4],
#     "MIN_SAMPLES": [1, 2, 3, 4],
#     "CLUSTER_SELECTION_EPS": [0.0001, 0.001, 0.01, 0.1]
# }
# param_combinations = list(ParameterGrid(param_grid))
# for X_norm, name in [(e5_X_norm, "e5"), (sb_X_norm, "securebert"), (glove_X_norm, "glove")]:
#     results = Parallel(n_jobs=-1, verbose=5)(
#         delayed(evaluate_hdbscan)(X_norm, p["MIN_CLUSTER_SIZE"], p["MIN_SAMPLES"], p["CLUSTER_SELECTION_EPS"])
#         for p in param_combinations
#     )
#     pd.DataFrame(results).to_csv(f"../results/{name}_hybrid_tuning_results.csv", index=False)

# Load saved tuning results
e5_tuning = pd.read_csv("../results/e5_hybrid_tuning_results.csv")
sb_tuning = pd.read_csv("../results/securebert_hybrid_tuning_results.csv")
glove_tuning = pd.read_csv("../results/glove_hybrid_tuning_results.csv")

print("Hybrid E5 — Top 5 configurations:")
display(e5_tuning.sort_values("n_clusters", ascending=False).head(5))

print("\nHybrid SecureBERT — Top 5 configurations:")
display(sb_tuning.sort_values("n_clusters", ascending=False).head(5))

print("\nHybrid GloVe — Top 5 configurations:")
display(glove_tuning.sort_values("n_clusters", ascending=False).head(5))

## 8. Notable Cluster Inspection — Hybrid E5

Manual inspection of Hybrid E5 clusters confirmed event-level coherence across
multiple real-world cybersecurity incidents. The following clusters were verified
against known events using shared CVE identifiers and cross-outlet coverage.

In [ ]:
# Inspect specific clusters identified during manual review
notable_clusters = {
    207: "TikTok service shutdown — cross-outlet coverage",
    221: "Palo Alto Networks CVE-2024-3400 — including proof-of-concept disclosure",
    227: "BlackByte ransomware attack campaign",
    185: "Google security update — three CVEs verified via shared identifiers"
}

for cluster_id, description in notable_clusters.items():
    cluster_articles = e5_clustered[e5_clustered["cluster_id"] == cluster_id]
    print(f"\nCluster {cluster_id} — {description}")
    print(f"Articles in cluster: {len(cluster_articles)}")
    if "event_text" in cluster_articles.columns:
        for _, row in cluster_articles.head(3).iterrows():
            print(f"  → {str(row['event_text'])[:120]}...")

## 9. UMAP Visualization

UMAP projects the high-dimensional embedding space into 2D for visual inspection.
Each model is shown in two versions: **with noise** (all articles including those
HDBSCAN flagged as noise) and **without noise** (clustered articles only).

The no-noise views make cluster structure clearest. Hybrid E5 shows the strongest
separation — distinct event clusters scattered across UMAP space rather than
collapsed into a single dense mass.

These projections were generated from embeddings produced on the
CU Boulder Alpine HPC cluster.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

umap_images = [
    ("../results/e5_hybrid_no_noise.png",          "E5 Hybrid — No Noise"),
    ("../results/e5_hybrid_with_noise.png",         "E5 Hybrid — With Noise"),
    ("../results/securebert_hybrid_no_noise.png",   "SecureBERT Hybrid — No Noise"),
    ("../results/securebert_hybrid_with_noise.png", "SecureBERT Hybrid — With Noise"),
    ("../results/glove_hybrid_no_noise.png",        "GloVe Hybrid — No Noise"),
    ("../results/glove_hybrid_with_noise.png",      "GloVe Hybrid — With Noise"),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))

for ax, (path, title) in zip(axes.flatten(), umap_images):
    img = mpimg.imread(path)
    ax.imshow(img)
    ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    ax.axis("off")

plt.suptitle(
    "Hybrid Approach — UMAP Projections Across All Models",
    fontsize=14, fontweight="bold", y=1.01
)
plt.tight_layout()
plt.show()

### Key Visual Observations

**E5 Hybrid:** Distinct clusters scattered across UMAP space with clear separation
between groups. The no-noise view reveals fine-grained event-level structure —
individual incidents form tight, isolated groupings rather than blending into
a single dense mass.

**SecureBERT Hybrid:** A dominant central mass with scattered peripheral clusters.
The domain-adapted vocabulary improves over the Direct Approach but overweights
shared cybersecurity terminology, pulling many articles into the same dense region.

**GloVe Hybrid:** Two large dominant blobs with sparse outliers. The structural
template groupings are visible but correspond to rhetorical patterns (e.g., CISA
advisory format) rather than discrete incidents. GloVe's static word vectors lack
the contextual awareness to distinguish events sharing the same vocabulary.

**Conclusion:** Hybrid E5 is the only model that consistently separates articles
by incident rather than by topic, template, or vocabulary.

## 10. Results Summary

In [ ]:
summary = pd.DataFrame([
    {
        "Model": "Hybrid E5",
        "Clusters": e5_n,
        "Noise Fraction": round(e5_noise, 4),
        "Silhouette": round(e5_sil, 4) if e5_sil else "N/A",
        "Event-Level Coherence": "✓ Verified across multiple real incidents"
    },
    {
        "Model": "Hybrid SecureBERT",
        "Clusters": sb_n,
        "Noise Fraction": round(sb_noise, 4),
        "Silhouette": round(sb_sil, 4) if sb_sil else "N/A",
        "Event-Level Coherence": "Topical — sensitive to lexical repetition"
    },
    {
        "Model": "Hybrid GloVe",
        "Clusters": glove_n,
        "Noise Fraction": round(glove_noise, 4),
        "Silhouette": round(glove_sil, 4) if glove_sil else "N/A",
        "Event-Level Coherence": "Structural — groups by rhetorical template"
    }
])

print("Hybrid Approach — Final Results")
display(summary)

## 11. Conclusion

The Hybrid Approach succeeds where the Direct Approach failed. Dependency parsing
acts as pseudo-annotation — injecting relational structure into the embedding
pipeline without requiring manually labeled data.

**Key findings:**

**Hybrid E5** is the strongest performer. Its instruction-tuned training objective,
optimized for clustering and retrieval, produces the most directionally consistent
embeddings when applied to structured event templates. It successfully identifies
and groups real-world cybersecurity incidents across multiple news outlets —
including distinct clusters for CVE disclosures, proof-of-concept releases,
and ransomware campaigns.

**Hybrid SecureBERT** improves over its Direct Approach counterpart but remains
sensitive to surface lexical patterns. Domain adaptation improves terminology
recognition but risks overweighting shared vocabulary over shared events.

**Hybrid GloVe** captures structural patterns — grouping articles that share
rhetorical templates — but lacks the contextual awareness to distinguish
semantically similar events reliably.

**The central finding:** an unsupervised pathway for cybersecurity event extraction
is viable, but its performance depends critically on injecting event structure
prior to embedding. Holistic semantic embeddings, regardless of model quality
or domain adaptation, are insufficient for this task.